# HW 1 - Volatility of SPY (MA, EWMA, GARCH(1,1))

## Task 1 - MA(100) and EWMA(0.94) annualized volatility

**Pseudocode**

1. Set $n = 100$, $\lambda = 0.94$, $\Delta = 1/252$.
2. Download the daily close prices of SPY from Yahoo Finance, $\{s_0, s_1, \dots, s_N\}$, for 08/01/2021 - 07/30/2026.
3. Compute the log returns $x_i = \log(s_i / s_{i-1})$, $i = 1, \dots, N$.
4. Compute the daily volatility by MA($n$): for $i = n, \dots, N$
   $$\hat\mu_i = \frac{1}{n}\sum_{j=0}^{n-1} x_{i-j}, \qquad \hat\sigma_i^2 = \frac{1}{n-1}\sum_{j=0}^{n-1}(x_{i-j} - \hat\mu_i)^2 .$$
5. Compute the daily volatility by EWMA($\lambda$): set $\hat\sigma_{n}^2$ = sample variance of the first $n$ log returns, then for $i = n+1, \dots, N$
   $$\hat\sigma_i^2 = \lambda \hat\sigma_{i-1}^2 + (1-\lambda)\, x_{i-1}^2 .$$
6. Annualize both by multiplying by $\sqrt{1/\Delta} = \sqrt{252}$.
7. Plot the annualized MA($n$) and EWMA($\lambda$) estimates on the same figure.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
from scipy.optimize import minimize

# 1. parameters
n = 100
lam = 0.94
delta = 1 / 252

# 2. download prices (end is exclusive -> 2026-07-31 includes 2026-07-30)
prices = yf.download("SPY", start="2021-08-01", end="2026-07-31", progress=False)["Close"].squeeze()

# 3. log returns
x = np.log(prices / prices.shift(1)).dropna()

In [ ]:
# 4. MA(n) daily volatility
ma_vol = x.rolling(n).std()

# 5. EWMA daily volatility (initial variance = sample variance of the first n returns)
ewma_var = pd.Series(np.nan, index=x.index)
ewma_var.iloc[n - 1] = x.iloc[:n].var()
for i in range(n, len(x)):
    ewma_var.iloc[i] = lam * ewma_var.iloc[i - 1] + (1 - lam) * x.iloc[i - 1] ** 2
ewma_vol = np.sqrt(ewma_var)

# 6. annualize
ma_ann = ma_vol / np.sqrt(delta)
ewma_ann = ewma_vol / np.sqrt(delta)

In [ ]:
# 7. plot
plt.figure(figsize=(12, 6))
plt.plot(ma_ann, label=f"MA (n={n})")
plt.plot(ewma_ann, label=f"EWMA (lambda={lam})")
plt.xlabel("Date")
plt.ylabel("Annualized volatility")
plt.title("SPY annualized volatility: MA vs EWMA")
plt.legend()
plt.show()

## Task 2 - Fit GARCH(1,1) by MLE (06/01/2025 - 05/31/2026)

**Pseudocode**

Model: $\sigma_i^2 = \alpha_0 + \alpha_1 x_{i-1}^2 + \beta_1 \sigma_{i-1}^2$, $x_i =\sigma_i Z_i$, $Z_i \sim N(0,1)$ i.i.d.

1. Take the log returns $x_1, \dots, x_N$ from 06/01/2025 to 05/31/2026.
2. Define the negative log-likelihood $\theta = (\alpha_0, \alpha_1, \beta_1) \mapsto -\ell(\theta)$:
   1. Set $\sigma_1^2$ = sample variance of the log returns.
   2. For $i = 2, \dots, N$: $\sigma_i^2 = \alpha_0 + \alpha_1 x_{i-1}^2 + \beta_1 \sigma_{i-1}^2$.
   3. Return $\sum_{i=1}^{N}\left(\ln \sigma_i^2 + x_i^2/\sigma_i^2\right)$.
3. Set $\hat\theta = \arg\min_\theta -\ell(\theta)$ subject to $\alpha_0 > 0$, $\alpha_1, \beta_1 \ge 0$, $\alpha_1 + \beta_1 < 1$.

(Returns are multiplied by 100 during the optimization for numerical stability, and $\alpha_0$ is converted back at the end.)

In [ ]:
# 1. returns in the fitting window (in %)
xf = x.loc["2025-06-01":"2026-05-31"].values * 100
N = len(xf)

# 2. negative log-likelihood
def garch_variances(theta, xf):
    a0, a1, b1 = theta
    var = np.empty(len(xf))
    var[0] = xf.var(ddof=1)
    for i in range(1, len(xf)):
        var[i] = a0 + a1 * xf[i - 1] ** 2 + b1 * var[i - 1]
    return var

def neg_loglik(theta, xf):
    var = garch_variances(theta, xf)
    return np.sum(np.log(var) + xf ** 2 / var)

# 3. minimize
res = minimize(
    neg_loglik, x0=[0.05, 0.05, 0.90], args=(xf,),
    bounds=[(1e-8, None), (0, 1), (0, 1)],
    constraints={"type": "ineq", "fun": lambda t: 0.9999 - t[1] - t[2]},
    method="SLSQP",
)
a0, a1, b1 = res.x
print(f"alpha0 = {a0 / 1e4:.4e}  (= {a0:.4f} in %^2)")
print(f"alpha1 = {a1:.4f}")
print(f"beta1  = {b1:.4f}")
print(f"long-run annualized vol = {np.sqrt(a0 / (1 - a1 - b1) / 1e4 / delta):.4f}")

## Task 3 - Forecast with $M = 100$ simulated paths (06/01/2026 - 07/30/2026)

**Pseudocode**

1. Set $M = 100$ and let $m$ = number of trading days in 06/01/2026 - 07/30/2026.
2. Take the fitted $\hat\theta = (\hat\alpha_0, \hat\alpha_1, \hat\beta_1)$; set $\sigma_N^2$ = last fitted conditional variance and $x_N$ = last log return of the fitting window.
3. For each path $k = 1, \dots, M$:
   1. For $j = 1, \dots, m$:
      1. $\sigma_{N+j}^2 = \hat\alpha_0 + \hat\alpha_1 x_{N+j-1}^2 + \hat\beta_1 \sigma_{N+j-1}^2$.
      2. Draw $Z_{N+j} \sim N(0,1)$ and set $x_{N+j} = \sigma_{N+j} Z_{N+j}$.
   2. Store the path $\{\sigma_{N+j}\}_{j=1}^m$.
4. Annualize by $\sqrt{1/\Delta}$ and plot the $M$ paths together with the MA and EWMA estimates over the same period.

In [ ]:
# 1. setup
M = 100
dates = x.loc["2026-06-01":"2026-07-30"].index
m = len(dates)
rng = np.random.default_rng(0)

# 2. starting point
var_N = garch_variances(res.x, xf)[-1]
x_N = xf[-1]

# 3. simulate M paths
paths = np.empty((M, m))
for k in range(M):
    var_prev, x_prev = var_N, x_N
    for j in range(m):
        var_j = a0 + a1 * x_prev ** 2 + b1 * var_prev
        x_j = np.sqrt(var_j) * rng.standard_normal()
        paths[k, j] = np.sqrt(var_j)
        var_prev, x_prev = var_j, x_j

# 4. back to decimals and annualize
paths_ann = paths / 100 / np.sqrt(delta)

In [ ]:
plt.figure(figsize=(12, 6))
for k in range(M):
    plt.plot(dates, paths_ann[k], color="gray", alpha=0.15)
plt.plot([], [], color="gray", label="GARCH(1,1) simulated paths")
plt.plot(dates, ma_ann.loc[dates], color="blue", lw=2, label=f"MA (n={n})")
plt.plot(dates, ewma_ann.loc[dates], color="red", lw=2, label=f"EWMA (lambda={lam})")
plt.xlabel("Date")
plt.ylabel("Annualized volatility")
plt.title("GARCH(1,1) forecast paths vs MA and EWMA (06/01/2026 - 07/30/2026)")
plt.legend()
plt.show()

### Comparison and comments

- **GARCH paths.** All 100 simulated paths start near 11% and drift slowly upward towards the fitted long-run level (about 12%); the average path is about 11.8%, and 90% of the path values lie between roughly 10.7% and 13.4%. The forecast is smooth in the mean because, with $\hat\alpha_1 + \hat\beta_1 \approx 0.88 < 1$, today's variance is forgotten geometrically and mean-reverts to $V_L$. The spread between paths only reflects the random shocks $Z_{N+j}$ that we simulate.
- **MA(100).** It stays almost flat at about 13.5%-15% (average about 14.4%), above almost all simulated paths (about 99% of path-days are below it). The 100-day window still contains the higher-volatility days from spring 2026, so it reacts slowly and lags behind.
- **EWMA(0.94).** It is the most reactive: it starts near 11%, in line with GARCH, jumps to about 16% in mid-June after a few large realized returns, and then decays back to about 12%-13% by the end of July (average about 13.9%). This is the same kind of dynamics as GARCH(1,1) (EWMA is the case $\alpha_0 = 0$, $\alpha_1 + \beta_1 = 1$), but EWMA uses the *realized* returns of the period, while the GARCH paths do not.
- **Conclusion.** The GARCH forecast is a model-based, mean-reverting projection: it cannot anticipate the June volatility spike, which is why EWMA and MA sit above the bulk of the paths for most of the period, but the EWMA estimate falls back inside the GARCH band at the beginning and at the end of the window. MA and EWMA are backward-looking estimates of realized volatility, while GARCH gives a forward-looking distribution of possible volatility paths, so the fan of paths is the more informative object for risk purposes.